# Jev

[Jev](https://docs.typesafe.ai) is TypeSafe's decision model. It doesn't write text: you give it a **state** (text or JSON) and typed **questions**, and it returns each answer with the probability of every option. The answers can't leave the schema, and the confidence is measured, not self-reported.

`JevEvaluator` runs it in one of two places, with the same input and output:

- **Hosted** (default): TypeSafe's API. Set `TYPESAFE_API_KEY` (console.typesafe.ai -> API Keys).
- **Local** (`use_local_model=True`): [Strands Decider](https://github.com/strands-labs/strands-decider), an open-weights model with the same API, on your machine.

```bash
pip install "grounded-ai[jev]"        # hosted
pip install "grounded-ai[jev-local]"  # local
```

## Setup

Hosted when a key is set, local otherwise. `ask()` sends one request and returns its answers; if the call fails it raises with the reason (e.g. `429: rate limited`) instead of failing later on a missing `.answers`.

In [1]:
import os

from grounded_ai import Evaluator
from grounded_ai.backends.jev import (
    HALLUCINATION,
    RAG_RELEVANCE,
    TOXICITY,
    ChoiceQuestion,
    JevInput,
    NoulQuestion,
    ScoreQuestion,
)
from grounded_ai.schemas import EvaluationError

if os.getenv("TYPESAFE_API_KEY"):
    evaluator = Evaluator("jev/jev-latest")  # TypeSafe's hosted Jev
else:
    evaluator = Evaluator("jev", use_local_model=True)  # Strands Decider on this machine
    evaluator.backend.warmup(port=8000)  # starts the server; the first run downloads the model


def ask(input_data):
    result = evaluator.evaluate(input_data)
    if isinstance(result, EvaluationError):
        raise RuntimeError(f"{result.error_code}: {result.message}")
    return result.answers


print("Answering from", evaluator.backend.base_url)

Answering from https://api.typesafe.ai


## Built-in checks

Ready-made questions for the evaluations grounded-ai ships. Each reads named fields from the state: `HALLUCINATION` checks a `response` against its `context`, `TOXICITY` judges a `response`, and `RAG_RELEVANCE` asks whether a retrieved `context` can answer the `query`.

In [2]:
context = (
    "Apollo 11 landed on the Moon in 1969. Neil Armstrong and Buzz Aldrin walked on the surface "
    "while Michael Collins stayed in orbit."
)
checks = [
    ("hallucination", HALLUCINATION, {"context": context, "response": "Michael Collins stayed in orbit."}),
    ("hallucination", HALLUCINATION, {"context": context, "response": "Buzz Aldrin stayed in orbit while Collins walked on the Moon."}),
    ("toxicity", TOXICITY, {"response": "Thanks for the quick reply, that fixed it."}),
    ("toxicity", TOXICITY, {"response": "You are a worthless idiot and everyone hates you."}),
    ("rag relevance", RAG_RELEVANCE, {"query": "What does vitamin D do for the body?", "context": "Vitamin D helps the body absorb calcium, which keeps bones strong."}),
    ("rag relevance", RAG_RELEVANCE, {"query": "What does vitamin D do for the body?", "context": "Vitamin C is found in citrus fruits such as oranges and lemons."}),
]
for name, question, state in checks:
    answer = ask(JevInput(state=state, questions={"check": question}))["check"]
    print(f"{name:14} {answer.choice:14} confidence {answer.confidence:.2f}")

hallucination  faithful       confidence 1.00


hallucination  hallucination  confidence 1.00
toxicity       non-toxic      confidence 1.00


toxicity       toxic          confidence 1.00
rag relevance  relevant       confidence 1.00


rag relevance  unrelated      confidence 1.00


## Your own questions

Three question types, any number per request. One support ticket, three questions:

| Question | You give | You get |
| :--- | :--- | :--- |
| `NoulQuestion` | a yes/no question | `.noul`: probability of yes |
| `ChoiceQuestion` | options with descriptions | `.choice`, `.probabilities`, `.confidence` |
| `ScoreQuestion` | ordered levels, lowest first | `.score` (level index from 0), `.confidence` |

In [3]:
ticket = "I was charged twice this month and now my account is overdrawn. Reverse it today or I'm cancelling."

answers = ask(JevInput(
    state=ticket,
    questions={
        "urgent": NoulQuestion(instructions="Does the customer need a reply today?"),
        "team": ChoiceQuestion(
            instructions="Which team should handle this ticket?",
            criteria={
                "billing": "charges, refunds, invoices",
                "technical": "bugs, outages, integrations",
                "sales": "pricing, upgrades, new accounts",
            },
        ),
        "mood": ScoreQuestion(instructions="How upset is the customer?", criteria=["calm", "annoyed", "angry"]),
    },
))

print(f"urgent: p(yes) = {answers['urgent'].noul:.2f}")
print(f"team:   {answers['team'].choice} (confidence {answers['team'].confidence:.2f})")
print(f"mood:   {answers['mood'].score:.1f} on 0 = calm .. 2 = angry (confidence {answers['mood'].confidence:.2f})")

urgent: p(yes) = 0.91
team:   billing (confidence 1.00)
mood:   2.0 on 0 = calm .. 2 = angry (confidence 1.00)


## Your own input class

Subclass `JevInput` and add fields. By default they are sent as JSON in the order you declare them; override `build_state()` to render them your own way, here to put a rule in front of the text being judged.

In [4]:
class PortPolicy(JevInput):
    text: str

    def build_state(self):
        return f"Rule: services must only listen on port 443.\nText: {self.text}"


follows_rule = ChoiceQuestion(
    instructions="Does the text follow the rule?",
    criteria={"follows": "the text obeys the rule", "violates": "the text breaks the rule"},
)
for text in ["The service listens on 443.", "The API endpoint defaults to port 8080."]:
    answer = ask(PortPolicy(text=text, questions={"verdict": follows_rule}))["verdict"]
    print(f"{answer.choice:9} {text}")

follows   The service listens on 443.


violates  The API endpoint defaults to port 8080.


## Bad input never reaches the API

Every request is checked against the model's contract before it is sent. A state that isn't text or JSON comes back as an `INVALID_REQUEST` error, and nothing goes over the network.

In [5]:
class Broken(JevInput):
    def build_state(self):
        return 42  # not text or JSON


error = evaluator.evaluate(Broken(questions={"verdict": follows_rule}))
print(error.error_code)

INVALID_REQUEST


## Shut down

Stops the local server `warmup()` started. Hosted Jev has nothing to stop.

In [6]:
if evaluator.backend.use_local_model:
    evaluator.backend.shutdown()